## p50 Little Image denoising exermple

```
conda activate sora
```

In [ ]:
# PROMPT = "a gray tabby cat sitting on a windowsill, golden hour light, photorealistic"

In [ ]:
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
from pathlib import Path

PROMPT = "A happy gray tabby cat looking at the camera, photorealistic, 8k"
NEGATIVE_PROMPT = "cartoon, illustration, painting, drawing, anime, low quality"
SEED = 5 #12 is ok
NUM_STEPS = 100
GUIDANCE_SCALE = 7.0
OUTPUT_DIR = Path("data/robots/mar_17_2")
OUTPUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",
    torch_dtype=torch.float16,
)
# Better scheduler for photorealism
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="dpmsolver++",
)
pipe.to("cuda")

intermediate_latents = []

def capture_latents(pipe, step_index, timestep, callback_kwargs):
    intermediate_latents.append(callback_kwargs["latents"].detach().clone())
    return callback_kwargs

generator = torch.Generator(device="cuda").manual_seed(SEED)

result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    num_inference_steps=NUM_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    callback_on_step_end=capture_latents,
)

final_image = result.images[0]
final_image.save(OUTPUT_DIR / "final.png")

# --- Decode intermediates (SD1.5 VAE is tiny, no VRAM issues) ---
step_images = []
for i, latents in enumerate(intermediate_latents):
    with torch.no_grad():
        decoded = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[0]
        decoded = (decoded / 2 + 0.5).clamp(0, 1)
        img = decoded[0].permute(1, 2, 0).cpu().float().numpy()
        img = Image.fromarray((img * 255).astype("uint8"))
    step_images.append(img)
    img.save(OUTPUT_DIR / f"step_{i:03d}.png")

print(f"Done — {len(step_images)} steps to {OUTPUT_DIR}/")

In [ ]:
final_image

---

In [ ]:
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload() #save some VRAM by offloading the model to CPU. Remove this if you have enough GPU power

prompt = "A cat holding a sign that says hello world"
image = pipe(
    prompt,
    height=1024,
    width=1024,
    guidance_scale=3.5,
    num_inference_steps=50,
    max_sequence_length=512,
    generator=torch.Generator("cpu").manual_seed(0)
).images[0]
image.save("flux-dev.png")


In [ ]:
from diffusers import FluxPipeline, FluxTransformer2DModel
from transformers import BitsAndBytesConfig
import torch

nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load transformer quantized, then plug it into the pipeline
transformer = FluxTransformer2DModel.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    subfolder="transformer",
    quantization_config=nf4_config,
    torch_dtype=torch.bfloat16,
)

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    transformer=transformer,
    torch_dtype=torch.bfloat16,
)
pipe.to("cuda")

In [ ]:
# import torch
# from diffusers import FluxPipeline
# from PIL import Image
# from pathlib import Path

# --- Config ---
PROMPT = "a gray tabby cat sitting on a windowsill, golden hour light, photorealistic"
SEED = 42
NUM_STEPS = 30
GUIDANCE_SCALE = 3.5  # FLUX likes lower guidance than SD
OUTPUT_DIR = Path("data/robots/mar_17_1")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# --- Load pipeline ---
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16,  # FLUX prefers bf16
)
# pipe.enable_model_cpu_offload()

# pipe.to("cuda")

pipe.enable_model_cpu_offload()
# If tight on VRAM, uncomment:
# pipe.enable_model_cpu_offload()


In [ ]:
# --- Capture intermediates ---
intermediate_latents = []

def capture_latents(pipe, step_index, timestep, callback_kwargs):
    intermediate_latents.append(callback_kwargs["latents"].detach().clone())
    return callback_kwargs

In [ ]:
# --- Generate ---
generator = torch.Generator(device="cuda").manual_seed(SEED)

result = pipe(
    prompt=PROMPT,
    num_inference_steps=NUM_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    callback_on_step_end=capture_latents,
)

final_image = result.images[0]
final_image.save(OUTPUT_DIR / "final.png")

In [ ]:
final_image

In [ ]:
# --- Decode intermediates ---
print(f"Decoding {len(intermediate_latents)} steps...")

step_images = []
for i, latents in enumerate(intermediate_latents):
    with torch.no_grad():
        # FLUX uses the same VAE decode pattern
        latents_scaled = (latents / pipe.vae.config.scaling_factor) + pipe.vae.config.shift_factor
        decoded = pipe.vae.decode(latents_scaled, return_dict=False)[0]
        decoded = (decoded / 2 + 0.5).clamp(0, 1)
        img = decoded[0].permute(1, 2, 0).cpu().float().numpy()
        img = Image.fromarray((img * 255).astype("uint8"))

    step_images.append(img)
    img.save(OUTPUT_DIR / f"step_{i:03d}.png")

print(f"Done — {len(step_images)} steps saved to {OUTPUT_DIR}/")